In [1]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)
np.random.seed(42)

d = np.load("../data/processed/windows_fd002.npz")
RUL_CAP = 125.0

def to_tensors(X, y):
    # targets are divided by 125 so the model learns values between 0 and 1
    return torch.tensor(X, dtype=torch.float32), torch.tensor(y / RUL_CAP, dtype=torch.float32)

X_tr, y_tr = to_tensors(d["X_tr"], d["y_tr"])
X_val, y_val = to_tensors(d["X_val"], d["y_val"])
X_test = torch.tensor(d["X_test"], dtype=torch.float32)

y_val_np = d["y_val"]
y_test_true = d["y_test"]
y_test_cap = np.minimum(y_test_true, RUL_CAP)

train_loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)


class LSTMRegressor(nn.Module):
    def __init__(self, n_features=28, hidden=64, layers=2, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(n_features, hidden, num_layers=layers,
                            batch_first=True, dropout=dropout)
        self.head = nn.Sequential(nn.Linear(hidden, 32), nn.ReLU(), nn.Linear(32, 1))

    def forward(self, x):
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :]).squeeze(-1)


def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

def nasa_score(y_true, y_pred):
    diff = y_pred - y_true
    return float(np.sum(np.where(diff < 0, np.exp(-diff / 13) - 1, np.exp(diff / 10) - 1)))

def predict(model, X):
    model.eval()
    with torch.no_grad():
        return model(X).numpy() * RUL_CAP


model = LSTMRegressor()
xb, yb = next(iter(train_loader))
print(xb.shape, yb.shape, model(xb).shape)
print("Parameters:", sum(p.numel() for p in model.parameters()))
print("Windows:", len(X_tr), "train,", len(X_val), "val,", len(X_test), "test")

torch.Size([128, 30, 28]) torch.Size([128]) torch.Size([128])
Parameters: 59457
Windows: 37174 train, 9045 val, 259 test


In [2]:
import time

model = LSTMRegressor()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

EPOCHS = 30
best_val, best_state = float("inf"), None
t0 = time.time()

for epoch in range(1, EPOCHS + 1):
    model.train()
    for xb, yb in train_loader:
        optimizer.zero_grad()
        loss_fn(model(xb), yb).backward()
        optimizer.step()
    val_rmse = rmse(y_val_np, predict(model, X_val))
    if val_rmse < best_val:
        best_val = val_rmse
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    print(f"epoch {epoch:2d}  val RMSE {val_rmse:6.2f}  ({time.time() - t0:.0f}s)")

model.load_state_dict(best_state)
test_pred = predict(model, X_test)
print(f"\nBest val RMSE: {best_val:.1f}")
print(f"Test RMSE (capped): {rmse(y_test_cap, test_pred):.1f}")
print(f"Test RMSE (true):   {rmse(y_test_true, test_pred):.1f}")
print(f"Test score:         {nasa_score(y_test_true, test_pred):.0f}")

epoch  1  val RMSE  16.27  (11s)
epoch  2  val RMSE  16.45  (24s)
epoch  3  val RMSE  15.82  (40s)
epoch  4  val RMSE  15.47  (60s)
epoch  5  val RMSE  15.22  (82s)
epoch  6  val RMSE  14.97  (103s)
epoch  7  val RMSE  16.64  (124s)
epoch  8  val RMSE  15.83  (145s)
epoch  9  val RMSE  16.64  (166s)
epoch 10  val RMSE  16.38  (187s)
epoch 11  val RMSE  16.50  (208s)
epoch 12  val RMSE  16.57  (280s)
epoch 13  val RMSE  17.10  (300s)
epoch 14  val RMSE  18.03  (321s)
epoch 15  val RMSE  17.76  (342s)
epoch 16  val RMSE  17.64  (363s)
epoch 17  val RMSE  18.08  (384s)
epoch 18  val RMSE  18.35  (405s)
epoch 19  val RMSE  19.03  (426s)
epoch 20  val RMSE  18.97  (447s)
epoch 21  val RMSE  18.98  (468s)
epoch 22  val RMSE  18.88  (489s)
epoch 23  val RMSE  19.33  (510s)
epoch 24  val RMSE  19.58  (531s)
epoch 25  val RMSE  19.23  (553s)
epoch 26  val RMSE  19.37  (574s)
epoch 27  val RMSE  19.47  (595s)
epoch 28  val RMSE  18.75  (616s)
epoch 29  val RMSE  18.92  (636s)
epoch 30  val RMSE 

In [3]:
def train_reg(seed=42, epochs=40, lr=1e-3, wd=1e-4, dropout=0.3, patience=6, verbose=True):
    torch.manual_seed(seed)
    np.random.seed(seed)
    loader = DataLoader(TensorDataset(X_tr, y_tr), batch_size=128, shuffle=True)
    m = LSTMRegressor(dropout=dropout)
    opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    sched = torch.optim.lr_scheduler.ReduceLROnPlateau(opt, factor=0.5, patience=2)
    loss_fn = nn.MSELoss()
    best_v, best_s, bad = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        m.train()
        total, n = 0.0, 0
        for xb, yb in loader:
            opt.zero_grad()
            loss = loss_fn(m(xb), yb)
            loss.backward()
            opt.step()
            total += loss.item() * len(xb)
            n += len(xb)
        train_rmse = (total / n) ** 0.5 * RUL_CAP
        v = rmse(y_val_np, predict(m, X_val))
        sched.step(v)
        if verbose:
            print(f"epoch {epoch:2d}  train {train_rmse:5.1f}  val {v:5.1f}  lr {opt.param_groups[0]['lr']:.5f}")
        if v < best_v:
            best_v, bad = v, 0
            best_s = {k: t.clone() for k, t in m.state_dict().items()}
        else:
            bad += 1
            if bad >= patience:
                break

    m.load_state_dict(best_s)
    return m, best_v

model_reg, best_v = train_reg()
p = predict(model_reg, X_test)
print(f"\nBest val RMSE: {best_v:.1f}")
print(f"Test RMSE (capped): {rmse(y_test_cap, p):.1f}")
print(f"Test RMSE (true):   {rmse(y_test_true, p):.1f}")
print(f"Test score:         {nasa_score(y_test_true, p):.0f}")

epoch  1  train  27.2  val  16.1  lr 0.00100
epoch  2  train  16.5  val  17.1  lr 0.00100
epoch  3  train  15.4  val  15.1  lr 0.00100
epoch  4  train  14.6  val  15.1  lr 0.00100
epoch  5  train  13.9  val  15.5  lr 0.00100
epoch  6  train  13.5  val  15.1  lr 0.00100
epoch  7  train  13.0  val  16.5  lr 0.00050
epoch  8  train  12.1  val  16.0  lr 0.00050
epoch  9  train  11.7  val  16.7  lr 0.00050
epoch 10  train  11.4  val  16.4  lr 0.00025

Best val RMSE: 15.1
Test RMSE (capped): 14.2
Test RMSE (true):   27.2
Test score:         9902


In [4]:
rows = []
for seed in [1, 2, 3, 4, 5]:
    t0 = time.time()
    m, v = train_reg(seed=seed, verbose=False)
    p = predict(m, X_test)
    rows.append((v, rmse(y_test_cap, p), rmse(y_test_true, p), nasa_score(y_test_true, p)))
    print(f"seed {seed}: val {v:.1f}  test(capped) {rows[-1][1]:.1f}  test(true) {rows[-1][2]:.1f}  score {rows[-1][3]:.0f}  ({time.time() - t0:.0f}s)")

arr = np.array(rows)
print("\nMean +/- std over 5 seeds")
for i, name in enumerate(["val RMSE", "test (capped)", "test (true)", "score"]):
    print(f"{name:<14}{arr[:, i].mean():8.1f} +/- {arr[:, i].std():.1f}")

seed 1: val 15.2  test(capped) 15.2  test(true) 28.3  score 11278  (214s)
seed 2: val 14.8  test(capped) 14.7  test(true) 27.7  score 13390  (184s)
seed 3: val 15.0  test(capped) 14.4  test(true) 27.2  score 9096  (220s)
seed 4: val 15.0  test(capped) 15.0  test(true) 26.7  score 9512  (246s)
seed 5: val 14.7  test(capped) 15.0  test(true) 26.7  score 7414  (253s)

Mean +/- std over 5 seeds
val RMSE          14.9 +/- 0.2
test (capped)     14.8 +/- 0.3
test (true)       27.3 +/- 0.6
score          10138.2 +/- 2038.4


In [5]:
import json, os

os.makedirs("../results", exist_ok=True)
lstm_fd002 = {
    "lstm_regularized_5_seeds": {
        "seeds": [1, 2, 3, 4, 5],
        "val_rmse": [float(arr[:, 0].mean()), float(arr[:, 0].std())],
        "test_rmse_capped": [float(arr[:, 1].mean()), float(arr[:, 1].std())],
        "test_rmse_true": [float(arr[:, 2].mean()), float(arr[:, 2].std())],
        "test_score": [float(arr[:, 3].mean()), float(arr[:, 3].std())],
        "note": "each pair is [mean, std] over seeds; dropout 0.3, AdamW weight decay 1e-4, LR plateau schedule, early stopping patience 6",
    }
}
with open("../results/lstm_fd002_5seeds.json", "w") as f:
    json.dump(lstm_fd002, f, indent=2)

print(open("../results/lstm_fd002_5seeds.json").read())

{
  "lstm_regularized_5_seeds": {
    "seeds": [
      1,
      2,
      3,
      4,
      5
    ],
    "val_rmse": [
      14.938289451599122,
      0.19444735675953365
    ],
    "test_rmse_capped": [
      14.84021816253662,
      0.2795184457356572
    ],
    "test_rmse_true": [
      27.306958770751955,
      0.6260809579524966
    ],
    "test_score": [
      10138.1580078125,
      2038.4372897328374
    ],
    "note": "each pair is [mean, std] over seeds; dropout 0.3, AdamW weight decay 1e-4, LR plateau schedule, early stopping patience 6"
  }
}
